# Model Training Results

Compare cross-validation and validation performance for each task. The held-out test set is evaluated only for the selected model and is reported separately.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
with (ROOT / "config.yaml").open(encoding="utf-8") as file:
    project_config = yaml.safe_load(file)
graphics = project_config["graphics"]
plt.style.use(graphics["style"])
sns.set_palette(graphics["categorical_palette"])

results_path = ROOT / project_config["output"]["model_artifacts_path"] / "comparison.csv"
comparison = pd.read_csv(results_path)
comparison["selected"] = comparison["selected"].astype(str).str.lower().eq("true")
comparison.head()

## 1. Anomaly Detection

The chart compares mean cross-validation macro F1 with validation macro F1. The test score is shown only for the selected model.

In [ ]:
task_results = comparison[comparison["task_name"] == "binary_detection"].copy()
display(task_results.sort_values("validation_f1_macro", ascending=False))
plot_data = task_results.melt(
    id_vars="model_name",
    value_vars=["cv_f1_macro_mean", "validation_f1_macro"],
    var_name="split",
    value_name="macro_f1",
).dropna()
fig, ax = plt.subplots(figsize=tuple(graphics["figsize"]))
sns.barplot(data=plot_data, x="model_name", y="macro_f1", hue="split", ax=ax)
ax.set(title="Anomaly Detection: CV vs Validation", xlabel="Model", ylabel="Macro F1")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
display(task_results.loc[task_results["selected"], ["model_name", "test_f1_macro"]])

## 2. Type Classification

Cross-validation and validation scores are used for comparison; test metrics belong only to the selected model.

In [ ]:
task_results = comparison[comparison["task_name"] == "anomaly_type"].copy()
display(task_results.sort_values("validation_f1_macro", ascending=False))
plot_data = task_results.melt(
    id_vars="model_name",
    value_vars=["cv_f1_macro_mean", "validation_f1_macro"],
    var_name="split",
    value_name="macro_f1",
).dropna()
fig, ax = plt.subplots(figsize=tuple(graphics["figsize"]))
sns.barplot(data=plot_data, x="model_name", y="macro_f1", hue="split", ax=ax)
ax.set(title="Type Classification: CV vs Validation", xlabel="Model", ylabel="Macro F1")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
display(task_results.loc[task_results["selected"], ["model_name", "test_f1_macro"]])

## 3. Temporal Localization

Scores are point-wise across the sequence. The selected model's threshold is calibrated on validation, then kept unchanged for the final test evaluation.

In [ ]:
task_results = comparison[comparison["task_name"] == "temporal_localization"].copy()
display(task_results.sort_values("validation_f1_macro", ascending=False))
plot_data = task_results.melt(
    id_vars="model_name",
    value_vars=["cv_f1_macro_mean", "validation_f1_macro"],
    var_name="split",
    value_name="macro_f1",
).dropna()
fig, ax = plt.subplots(figsize=tuple(graphics["figsize"]))
sns.barplot(data=plot_data, x="model_name", y="macro_f1", hue="split", ax=ax)
ax.set(title="Temporal Localization: CV vs Validation", xlabel="Model", ylabel="Macro F1")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
display(task_results.loc[task_results["selected"], ["model_name", "test_f1_macro", "decision_thresholds"]])

## 4. Affected KPI

The multi-label comparison reports macro F1 (plus micro F1 and exact match in the table). Per-label thresholds are calibrated from validation predictions.

In [ ]:
task_results = comparison[comparison["task_name"] == "affected_kpis"].copy()
display(task_results.sort_values("validation_f1_macro", ascending=False))
plot_data = task_results.melt(
    id_vars="model_name",
    value_vars=["cv_f1_macro_mean", "validation_f1_macro"],
    var_name="split",
    value_name="macro_f1",
).dropna()
fig, ax = plt.subplots(figsize=tuple(graphics["figsize"]))
sns.barplot(data=plot_data, x="model_name", y="macro_f1", hue="split", ax=ax)
ax.set(title="Affected KPI: CV vs Validation", xlabel="Model", ylabel="Macro F1")
ax.tick_params(axis="x", rotation=20)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
display(task_results.loc[task_results["selected"], ["model_name", "test_f1_macro", "test_f1_micro"]])